# Stage 09 — Registry and the promotion gate

**Track A (Buse) · Stage 9 of 10**

| | |
|---|---|
| **Input** | Trained models, eval results |
| **Output** | `models/registry_B.json`, final thresholds in `eval/thresholds_B.yaml` |
| **Promotes to** | `src/research_assistant/reranker/registry_B.py`, `baseline_B.py` |
| **Consumed by** | Sude's `eval/run_gate_S.py` and the workflow that calls it |

## What the registry is for

It is the indirection that makes the reranker swap real. Retrieval asks the registry
for the active ranker by name. The registry resolves the name to either the naive
similarity ranking or a tuned cross-encoder directory. Nothing upstream of it, and
nothing in Sude's tools, knows which one answered.

Without this, promoting a model is a code change. With it, promoting a model is a
config change plus a gate run, which is the entire point of the CI/CD story in the report.

## Design choices

| Choice | Picked | Alternatives | Why |
|---|---|---|---|
| Registry format | A JSON file in the repo | MLflow Model Registry, a database | The gate needs to read it in CI without credentials or a running service. MLflow still holds the runs and metrics, this file just names the champion. |
| Selection | `rerank.active` in the retrieval config | Environment variable, always-latest | A config value is reviewable in a pull request. Always-latest means a bad model ships the moment it finishes training. |
| Rollback | Keep every entry, never overwrite | Replace on promote | Rollback should be editing one line, not retraining. |
| Gate comparison | Against the current champion, not only fixed floors | Fixed thresholds only | Fixed floors miss slow regressions where each change loses a little and all of them pass. |

In [ ]:
from _nbsetup_B import REPO, load_cfg, resolve
import json, datetime
from pathlib import Path

rcfg = load_cfg("reranker")
reg_path = resolve(rcfg["registry"]["manifest"])
reg_path.parent.mkdir(parents=True, exist_ok=True)

registry = json.loads(reg_path.read_text(encoding="utf-8")) if reg_path.exists() else {
    "champion": "baseline",
    "entries": {
        "baseline": dict(kind="similarity", path=None, base_model=None,
                         notes="fusion order from stage 04, no cross-encoder"),
    },
}
registry

In [ ]:
def register(name, path, base_model, mlflow_run_id, metrics, notes=""):
    registry["entries"][name] = dict(
        kind="cross_encoder", path=str(path), base_model=base_model,
        mlflow_run_id=mlflow_run_id, metrics=metrics, notes=notes,
        registered_at=datetime.datetime.now().isoformat(timespec="seconds"),
    )
    reg_path.write_text(json.dumps(registry, indent=2), encoding="utf-8")
    print("registered", name, "- champion is still", registry["champion"])

def promote(name):
    # Only after the gate passes. Promoting by hand defeats the gate.
    assert name in registry["entries"], name
    registry["champion"] = name
    reg_path.write_text(json.dumps(registry, indent=2), encoding="utf-8")
    print("champion ->", name)

# register("dpo_v1", resolve(rcfg["train"]["output_dir"]), rcfg["model"]["base"],
#          "TODO run id", dict(ndcg5=0.0, mrr=0.0, recall20=0.0, cite_prec=0.0))

## The handoff to Sude

Her gate runner reads three things you own, and nothing else:

1. `eval/datasets/queries_B.jsonl` and `eval/datasets/qrels_B.jsonl`, in the frozen schema.
2. `eval/thresholds_B.yaml`, for the floors and the regression tolerances.
3. `eval/metrics/retrieval_B.py`, for the metric functions.

Keep those three stable and her workflow never breaks because of your changes.
When you do need to change one, it is a conversation before a commit, because a
threshold change silently redefines what every past passing run meant.

## Exit checks

- [ ] Switching `rerank.active` between `baseline` and `dpo` changes results, and
      nothing outside the registry needed editing.
- [ ] The gate runs locally end to end and fails when you feed it a deliberately bad model.
- [ ] Rollback is one line in the registry, and you have tried it.
- [ ] Thresholds are the ones you defended in stage 06, and Sude knows the reasoning.